# Semana 3 — Clasificación de estados, distribución estacionaria y comportamiento límite

**Curso:** Procesos Estocásticos · Ingeniería Industrial · Universidad EIA  
**Bloque:** A — Procesos Estocásticos (Ross, cap. 4)  
**Resultado de aprendizaje:** RA1 — *Modelar sistemas estocásticos discretos y continuos aplicando principios de probabilidad e ingeniería.*  
**Tipo de notebook:** Contenido (no gestionado por nbgrader — todas las actividades de esta semana son formativas, sin nota).

---

**Objetivos de esta sesión:**

- **O1.** Clasificar los estados de una cadena: accesibilidad, comunicación, clases, recurrencia y transitoriedad.
- **O2.** Distinguir cadenas irreducibles de cadenas con estructura de clases, y reconocer periodicidad.
- **O3.** Calcular la distribución estacionaria resolviendo $\pi = \pi P$ con la condición de normalización.
- **O4.** Interpretar el comportamiento límite: cuándo $P^n$ converge, qué significa esa convergencia y cuándo **no** ocurre.

**Conexión con lo que viene:** en la Semana 2 observaste, en la Actividad 3, que al aumentar el número de pasos la probabilidad dejaba de depender del estado inicial. Hoy explicamos **por qué** pasa eso, **bajo qué condiciones** pasa, y **qué hacer cuando no pasa**. En el Taller 01 estimaste ese comportamiento simulando trayectorias; hoy llegamos al mismo número por álgebra lineal, en una fracción del tiempo de cómputo.

In [ ]:
import numpy as np
from numpy.linalg import matrix_power

np.set_printoptions(precision=4, suppress=True)

# La misma máquina de las Semanas 1 y 2: Operativa (0), Mantenimiento (1), Dañada (2)
estados = ["Operativa", "Mantenimiento", "Dañada"]

P = np.array([
    [0.90, 0.07, 0.03],
    [0.60, 0.35, 0.05],
    [0.10, 0.50, 0.40],
])

print("Matriz de transición P:")
print(P)

---

## 1. Accesibilidad y comunicación — *(O1)*

Antes de preguntarnos por el comportamiento de largo plazo, necesitamos saber si la cadena está "bien conectada". Dos definiciones bastan.

> **Accesibilidad.** El estado $j$ es **accesible** desde $i$ (se escribe $i \to j$) si existe algún $n \geq 0$ tal que $p_{ij}^{(n)} > 0$. Es decir: hay al menos un camino, de cualquier longitud, que lleva de $i$ a $j$.

> **Comunicación.** Los estados $i$ y $j$ **se comunican** (se escribe $i \leftrightarrow j$) si $i \to j$ y $j \to i$.

La comunicación es una relación de equivalencia: es reflexiva, simétrica y transitiva. Y eso tiene una consecuencia muy útil — **particiona el espacio de estados en clases de comunicación** que no se solapan.

En el grafo de la Semana 2, esto se lee directamente: dos estados se comunican si puedes ir de uno al otro siguiendo las flechas, y volver.

In [ ]:
# Un truco práctico: si (I + P)^(N-1) tiene todas sus entradas positivas,
# entonces todos los estados se comunican entre sí.
# Sumar la identidad permite "quedarse" en un estado, así que la potencia
# acumula todos los caminos de longitud 0 hasta N-1.

N = len(estados)
alcanzabilidad = matrix_power(np.eye(N) + P, N - 1) > 0

print("Matriz de alcanzabilidad (True = j es accesible desde i):")
print(alcanzabilidad)
print()
print("¿Todos los estados se comunican entre sí?", alcanzabilidad.all())

### Actividad 1 — Leer la estructura de clases *(O1, formativa)*

Considera esta cadena de 4 estados, distinta a la de la máquina:

$$
Q = \begin{pmatrix}
0.5 & 0.5 & 0 & 0 \\
0.3 & 0.7 & 0 & 0 \\
0.2 & 0.1 & 0.4 & 0.3 \\
0 & 0 & 0 & 1
\end{pmatrix}
$$

Sin correr código todavía, responde:

**a)** ¿Cuáles estados se comunican entre sí? ¿Cuántas clases hay?

**b)** Desde el estado 2, ¿puedes llegar al estado 0? ¿Y desde el 0 al 2?

**c)** ¿Qué tiene de particular el estado 3?

In [ ]:
# Verifica tu razonamiento
Q = np.array([
    [0.5, 0.5, 0.0, 0.0],
    [0.3, 0.7, 0.0, 0.0],
    [0.2, 0.1, 0.4, 0.3],
    [0.0, 0.0, 0.0, 1.0],
])

alcanzabilidad_Q = matrix_power(np.eye(4) + Q, 3) > 0
print("Matriz de alcanzabilidad de Q:")
print(alcanzabilidad_Q)

La matriz de alcanzabilidad ya responde la pregunta, pero la estructura de clases se lee mucho más rápido en el grafo. Retomamos `networkx` de la Semana 2, ahora coloreando cada nodo según la clase a la que pertenece.

La función `nx.strongly_connected_components` hace exactamente lo que definimos como clase de comunicación: agrupa los nodos entre los cuales existe camino de ida y de vuelta.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G_Q = nx.DiGraph()
for i in range(4):
    for j in range(4):
        if Q[i, j] > 0:
            G_Q.add_edge(i, j, weight=Q[i, j])

# Las clases de comunicación son las componentes fuertemente conexas del grafo
clases = list(nx.strongly_connected_components(G_Q))
print("Clases de comunicación:", clases)

paleta = ["lightblue", "lightcoral", "lightgreen", "khaki"]
color_de = {nodo: paleta[k % len(paleta)]
            for k, clase in enumerate(clases) for nodo in clase}

pos = nx.circular_layout(G_Q)
plt.figure(figsize=(7, 6))
nx.draw_networkx_nodes(G_Q, pos, node_color=[color_de[n] for n in G_Q.nodes()], node_size=2000)
nx.draw_networkx_labels(G_Q, pos)
nx.draw_networkx_edges(G_Q, pos, connectionstyle="arc3,rad=0.15", arrowsize=20)
etiquetas = {(u, v): f"{d['weight']:.1f}" for u, v, d in G_Q.edges(data=True)}
nx.draw_networkx_edge_labels(G_Q, pos, edge_labels=etiquetas, font_size=9)
plt.title("Cadena Q: cada color es una clase de comunicación")
plt.axis("off")
plt.show()

El grafo hace visible de inmediato lo que la matriz esconde: del bloque $\{0, 1\}$ **no sale ninguna flecha** hacia el resto, y hacia el nodo 2 **no entra ninguna**. Esa asimetría es justamente lo que rompe la comunicación.

Compáralo con el grafo de la máquina en la Semana 2, donde toda flecha tenía su recíproca disponible por algún camino — esa era la señal visual de irreducibilidad.

<details>
<summary><b>▶ Revelar solución y explicación</b></summary>

**a) Hay tres clases:** $\{0, 1\}$, $\{2\}$ y $\{3\}$.

Los estados 0 y 1 se comunican: desde 0 puedes ir a 1 ($q_{01}=0.5$) y desde 1 puedes volver a 0 ($q_{10}=0.3$). El estado 2 forma su propia clase: puede salir hacia 0, 1 y 3, pero **nadie regresa a él** (mira la columna 2: solo $q_{22}$ es positivo). El estado 3 también está solo.

**b) Sí puedes ir de 2 a 0** (directamente, $q_{20}=0.2$). **No puedes ir de 0 a 2** — mira la fila 0: solo hay probabilidad hacia 0 y 1. Como la accesibilidad no es mutua, 0 y 2 **no** se comunican.

**c) El estado 3 es absorbente:** $q_{33}=1$. Una vez que la cadena entra, no sale nunca. Es el caso extremo de una clase cerrada de un solo elemento.

**Por qué importa:** esta cadena **no es irreducible**, y eso cambia por completo su comportamiento de largo plazo. No hay una única distribución estacionaria: a dónde converge la cadena depende de dónde empezó. Volveremos a esto en la sección 4.

</details>

---

## 2. Recurrencia, transitoriedad e irreducibilidad — *(O1, O2)*

La estructura de clases nos permite clasificar cada estado por su destino de largo plazo.

> **Estado recurrente.** Si la cadena parte de $i$, la probabilidad de **regresar** a $i$ en algún momento es 1. Consecuencia: la cadena visita $i$ infinitas veces.

> **Estado transitorio.** La probabilidad de regresar a $i$ es menor que 1. Consecuencia: la cadena visita $i$ un número finito de veces y eventualmente lo abandona para siempre.

En el ejemplo de la Actividad 1, el estado 2 es **transitorio**: cada vez que la cadena está en 2, tiene 60% de probabilidad de salir hacia una clase de la que no puede regresar. Tarde o temprano sale, y no vuelve.

Dos hechos que conviene tener presentes:

1. **La recurrencia es una propiedad de clase.** Si un estado de una clase es recurrente, todos lo son. Lo mismo con la transitoriedad.
2. **En una cadena finita, no todos los estados pueden ser transitorios.** Si la cadena tiene que estar en algún lado, al menos una clase debe ser recurrente.

> **Cadena irreducible.** Una cadena es **irreducible** si todos sus estados forman una única clase de comunicación — es decir, desde cualquier estado se puede llegar a cualquier otro.

La máquina de nuestro ejemplo **es irreducible**: lo confirmamos arriba, la matriz de alcanzabilidad es toda `True`. Eso significa que los tres estados son recurrentes, y que la cadena no se "queda atrapada" en ninguna región del espacio de estados.

### Actividad 2 — Clasificar los estados de la máquina *(O1, O2, formativa)*

Para la matriz $P$ de la máquina:

**a)** ¿Cuántas clases de comunicación hay?

**b)** ¿Los estados son recurrentes o transitorios? Justifica usando el hecho de que la cadena es finita.

**c)** Si la máquina se dañara de forma irreparable (es decir, si $p_{22} = 1$ y el resto de la fila 2 fuera cero), ¿cómo cambiaría la clasificación de los estados 0 y 1?

In [ ]:
# Explora tu respuesta al literal (c): modifica la fila 2 y observa la alcanzabilidad
P_irreparable = P.copy()
P_irreparable[2] = [0.0, 0.0, 1.0]

print("P modificada (daño irreparable):")
print(P_irreparable)
print()
print("Alcanzabilidad:")
print(matrix_power(np.eye(3) + P_irreparable, 2) > 0)

<details>
<summary><b>▶ Revelar solución y explicación</b></summary>

**a) Una sola clase.** Todos los estados se comunican: la matriz de alcanzabilidad es enteramente `True`. La cadena es irreducible.

**b) Los tres son recurrentes.** En una cadena finita al menos una clase debe ser recurrente, y aquí solo hay una clase — luego esa clase es recurrente, y con ella todos sus estados. Intuitivamente: por más que la máquina se dañe, siempre hay camino de vuelta a Operativa, así que volverá infinitas veces.

**c) Cambia radicalmente.** Con $p_{22}=1$, el estado 2 se vuelve **absorbente**, y los estados 0 y 1 pasan a ser **transitorios**: desde ambos se puede llegar a 2, pero desde 2 no se regresa. La cadena termina, con probabilidad 1, atrapada en "Dañada".

Ese cambio de una sola fila convierte un modelo de operación sostenible en un modelo de vida útil hasta la falla — dos preguntas de ingeniería completamente distintas. Es un buen recordatorio de cuánta información hay codificada en la estructura de $P$, no solo en sus valores.

</details>

---

## 3. La distribución estacionaria — *(O3)*

Llegamos a la pregunta central de la semana.

**La pregunta que responde:** a largo plazo, ¿qué fracción del tiempo pasa la máquina en cada estado? Si mañana firmas un contrato de mantenimiento, ¿cuántos turnos al año esperas tenerla en reparación?

> **Distribución estacionaria.** Un vector de probabilidad $\pi = (\pi_0, \ldots, \pi_{N-1})$ es estacionario si
> $$\pi = \pi P \qquad \text{con} \qquad \sum_i \pi_i = 1, \quad \pi_i \geq 0$$

Léelo así: si la distribución de estados hoy es $\pi$, entonces mañana **también** será $\pi$. Es un punto fijo — la distribución que la cadena ya no modifica.

Nota la diferencia con $\pi_1 = \pi_0 P$ de la Semana 2: allí calculábamos cómo *cambia* la distribución; aquí buscamos la que **no cambia**.

### Cómo resolverlo

La ecuación $\pi = \pi P$ se reescribe como $\pi(P - I) = 0$, o transponiendo, $(P^T - I)\pi^T = 0$. Es un sistema homogéneo: $\pi^T$ es un vector propio izquierdo de $P$ asociado al valor propio 1.

El problema es que ese sistema tiene infinitas soluciones (cualquier múltiplo escalar sirve). Por eso la condición $\sum_i \pi_i = 1$ no es un detalle administrativo: es lo que selecciona **una** solución entre todas. En la práctica se implementa reemplazando una de las ecuaciones del sistema por la de normalización.

In [ ]:
def distribucion_estacionaria(P):
    """Resuelve pi = pi P con sum(pi) = 1, para una cadena irreducible.

    Construye el sistema (P^T - I) pi = 0 y reemplaza la última ecuación
    por la condición de normalización sum(pi) = 1.
    """
    N = P.shape[0]
    A = P.T - np.eye(N)
    A[-1, :] = 1.0          # última fila: la ecuación de normalización
    b = np.zeros(N)
    b[-1] = 1.0
    return np.linalg.solve(A, b)


pi = distribucion_estacionaria(P)

print("Distribución estacionaria π:")
for estado, p in zip(estados, pi):
    print(f"  {estado:15} {p:.4f}   ({p*100:.2f}% del tiempo)")
print()
print("Verificación — π debe sumar 1:", pi.sum())
print("Verificación — π P debe ser igual a π:", np.allclose(pi @ P, pi))

Interpretación en términos del problema: a largo plazo la máquina está operativa alrededor del 82% de los turnos, en mantenimiento cerca del 12.8%, y dañada aproximadamente el 5.2%. Sobre un año de 300 turnos, eso son unos 38 turnos de mantenimiento y 16 de daño — cifras con las que ya se puede dimensionar un contrato o un presupuesto de repuestos.

Y esto responde la pregunta que quedó abierta en el Taller 01: aquella estimación por simulación de miles de trayectorias converge a este mismo vector. La diferencia es que la simulación da una aproximación con error muestral, mientras que resolver el sistema lineal da el valor exacto en microsegundos.

### Actividad 3 — Estacionaria contra simulación *(O3, formativa)*

Simula una trayectoria larga de la cadena (por ejemplo 100 000 pasos), cuenta la fracción de tiempo que pasa en cada estado, y compara con $\pi$.

Luego responde: ¿qué tan larga tuvo que ser la simulación para acercarse a dos decimales? ¿Qué ventaja tiene entonces el método algebraico?

In [ ]:
rng = np.random.default_rng(42)

n_pasos = 100_000
estado_actual = 0        # arranca Operativa
conteos = np.zeros(3)

# YOUR CODE HERE
# Pista: en cada paso, usa rng.choice(3, p=P[estado_actual]) para elegir
# el siguiente estado, y acumula el conteo del estado visitado.

frecuencias = conteos / n_pasos
print("Frecuencias simuladas:", frecuencias)
print("Distribución π:       ", pi)

<details>
<summary><b>▶ Revelar solución y explicación</b></summary>

```python
for _ in range(n_pasos):
    conteos[estado_actual] += 1
    estado_actual = rng.choice(3, p=P[estado_actual])
```

Con 100 000 pasos las frecuencias coinciden con $\pi$ en aproximadamente tres decimales. Con 1 000 pasos la coincidencia es de apenas uno o dos, y varía notablemente entre corridas.

**La ventaja del método algebraico** es que entrega el valor exacto de inmediato, sin error muestral ni dependencia de la semilla. La simulación sigue siendo indispensable cuando el modelo es demasiado complejo para resolverse analíticamente —- colas con múltiples servidores, políticas de decisión, cadenas con espacio de estados enorme— pero cuando existe solución cerrada, usarla es preferible.

**Detalle conceptual:** que la fracción de tiempo en cada estado converja a $\pi$ no es una coincidencia numérica. Es el **teorema ergódico** para cadenas de Markov: en una cadena irreducible y recurrente positiva, el promedio temporal de una sola trayectoria converge al promedio sobre la distribución estacionaria. Es lo que justifica que simular una trayectoria larga sea un método válido de estimación.

</details>

---

## 4. Comportamiento límite: cuándo $P^n$ converge — *(O4)*

En la Semana 2 notaste que al aumentar $n$, las filas de $P^n$ se parecían cada vez más entre sí. Veámoslo explícitamente.

In [ ]:
for n in [1, 2, 5, 10, 30]:
    print(f"P^{n} =")
    print(matrix_power(P, n))
    print()

print("π (para comparar con las filas de P^30):")
print(pi)

Las filas de $P^{30}$ son prácticamente idénticas entre sí, y cada una coincide con $\pi$. Eso es exactamente lo que anticipaste en la Semana 2: **la influencia del estado inicial se desvanece**.

Formalmente:

> **Teorema del límite.** Si una cadena finita es **irreducible** y **aperiódica**, entonces
> $$\lim_{n \to \infty} p_{ij}^{(n)} = \pi_j \qquad \text{para todo } i$$
> es decir, $P^n$ converge a una matriz cuyas filas son todas iguales a $\pi$.

Las dos condiciones importan, y por razones distintas.

### Periodicidad

> **Período.** El período de un estado $i$ es el máximo común divisor de todos los $n$ para los cuales $p_{ii}^{(n)} > 0$. Si el período es 1, el estado es **aperiódico**.

Un atajo práctico: si algún $p_{ii} > 0$ —es decir, si el estado tiene un *self-loop* como los que viste en el grafo de la Semana 2— entonces ese estado es aperiódico de inmediato. Nuestra máquina cumple eso en los tres estados.

El caso periódico es más fácil de entender con un contraejemplo.

In [ ]:
# Cadena periódica: alterna forzosamente entre dos estados
R = np.array([
    [0.0, 1.0],
    [1.0, 0.0],
])

print("R   =", R.tolist())
print("R^2 =", matrix_power(R, 2).tolist())
print("R^3 =", matrix_power(R, 3).tolist())
print("R^4 =", matrix_power(R, 4).tolist())
print()
print("Distribución estacionaria de R:", distribucion_estacionaria(R))

Observa la disociación: $R^n$ **no converge** — oscila indefinidamente entre dos matrices según la paridad de $n$. Pero la distribución estacionaria $\pi = (0.5,\ 0.5)$ **sí existe** y es única.

La lectura correcta: $\pi$ sigue describiendo la fracción de tiempo a largo plazo (la cadena pasa la mitad del tiempo en cada estado, lo cual es cierto), pero **no** describe la probabilidad de estar en un estado en el paso $n$ (que depende de si $n$ es par o impar).

Conviene separar las tres afirmaciones, porque es fácil confundirlas:

| Condición | ¿Existe $\pi$ única? | ¿Converge $P^n$? |
|---|---|---|
| Irreducible y aperiódica | Sí | Sí |
| Irreducible y periódica | Sí | No |
| Reducible | En general no | Depende |

La tercera fila es la de la matriz $Q$ de la Actividad 1: con varias clases cerradas, el destino de largo plazo depende de la clase en la que la cadena termine, y por eso no hay una única $\pi$.

### Actividad 4 — Diagnóstico completo *(O2, O4, formativa)*

Para cada una de estas tres matrices, determina —razonando antes de correr código— si la cadena es irreducible, si es aperiódica, y si $P^n$ converge:

$$
A = \begin{pmatrix} 0.5 & 0.5 \\ 0.2 & 0.8 \end{pmatrix}
\qquad
B = \begin{pmatrix} 0 & 1 & 0 \\ 0 & 0 & 1 \\ 1 & 0 & 0 \end{pmatrix}
\qquad
C = \begin{pmatrix} 1 & 0 \\ 0.3 & 0.7 \end{pmatrix}
$$

In [ ]:
A = np.array([[0.5, 0.5], [0.2, 0.8]])
B = np.array([[0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [1.0, 0.0, 0.0]])
C = np.array([[1.0, 0.0], [0.3, 0.7]])

for nombre, M in [("A", A), ("B", B), ("C", C)]:
    print(f"--- {nombre} ---")
    print(f"M^20 =\n{matrix_power(M, 20)}")
    print(f"M^21 =\n{matrix_power(M, 21)}")
    print()

<details>
<summary><b>▶ Revelar solución y explicación</b></summary>

**$A$ — irreducible, aperiódica, converge.** Los dos estados se comunican y ambos tienen *self-loop*, así que el período es 1. $A^{20}$ y $A^{21}$ son idénticas: las filas convergieron a $\pi \approx (0.2857,\ 0.7143)$. Es el caso estándar, el mismo de la máquina.

**$B$ — irreducible, periódica de período 3, no converge.** Es un ciclo determinista $0 \to 1 \to 2 \to 0$. Todos los estados se comunican, pero solo se puede regresar a un estado en múltiplos de 3 pasos, así que el período es 3. $B^{20}$ y $B^{21}$ son matrices de permutación distintas. Aun así, $\pi = (1/3,\ 1/3,\ 1/3)$ existe y es correcta como fracción de tiempo.

**$C$ — reducible, converge, pero no por las razones del teorema.** El estado 0 es absorbente y el 1 es transitorio: no se comunican, luego la cadena no es irreducible. $C^{20}$ converge, pero a una matriz con filas $(1, 0)$ — la cadena termina absorbida en el estado 0 sin importar dónde empezó. Aquí la convergencia ocurre porque hay una única clase recurrente; con dos clases absorbentes, el límite dependería del estado inicial.

**La moraleja:** irreducible + aperiódica es una condición *suficiente* para la convergencia, no *necesaria*. Cuando no se cumple, hay que analizar la estructura de clases antes de sacar conclusiones.

</details>

---

## 5. Cierre y puente a la próxima semana

Esta semana cerramos el análisis de largo plazo de cadenas discretas. Clasificamos los estados por accesibilidad y comunicación, distinguimos recurrencia de transitoriedad, y establecimos qué significa que una cadena sea irreducible. Con eso pudimos plantear y resolver $\pi = \pi P$, y —más importante— entender **bajo qué condiciones** esa distribución describe efectivamente el comportamiento límite.

El punto que conviene retener: la distribución estacionaria casi siempre existe en cadenas irreducibles, pero solo describe el límite de $P^n$ cuando además hay aperiodicidad. Confundir ambas cosas es el error más frecuente al aplicar estos resultados.

También viste que la simulación del Taller 01 y el álgebra de hoy convergen al mismo vector. Esa equivalencia —promedio temporal de una trayectoria contra promedio sobre la distribución estacionaria— es el teorema ergódico, y es la base conceptual de buena parte de la simulación estocástica que veremos en el Bloque B.

La próxima semana pasamos del tiempo discreto al **tiempo continuo**: procesos de Poisson, tiempos entre eventos con distribución exponencial y la propiedad de pérdida de memoria en su versión continua. Es el paso natural para modelar llegadas a un sistema —clientes, fallas, solicitudes— donde los eventos no ocurren en turnos fijos sino en cualquier instante.

---
**Referencias de esta semana:** Ross, S. M. *Introduction to Probability Models*, cap. 4, secciones 4.3 (clasificación de estados) y 4.4 (probabilidades límite).